# 实验一：esmini 场景仿真——Cut-in 加塞测试

## 实验目标
- 理解 OpenSCENARIO 场景描述标准
- 用 Python 生成一个 cut-in（加塞）测试场景
- 用 esmini 无头模式回放场景，观察车辆行为

## 背景
Cut-in 是自动驾驶测试的经典场景：相邻车道的车辆突然切入自车前方，考验 AEB/ACC 系统的响应。
本实验用 `scenariogeneration` 生成标准的 OpenSCENARIO 1.1 文件，再用 esmini 播放验证。

In [ ]:
# 检查环境
import subprocess
r = subprocess.run(['/opt/esmini/esmini/bin/esmini', '--version'], capture_output=True, text=True)
print('esmini:', r.stdout.strip() or r.stderr.strip()[:60])
import scenariogeneration
print('scenariogeneration ok')

## 1. 生成场景
200m 双车道直道，Ego 在右车道 15m/s，Target 在左车道 12m/s，t=5s 时 Target 切入 Ego 车道。

In [ ]:
from scenariogeneration import xosc as OSC
from scenariogeneration.xodr import OpenDrive
from scenariogeneration.xodr.generators import create_straight_road

# 路网
road = create_straight_road(road_id=0, length=200, n_lanes=2, lane_offset=3)
odr = OpenDrive('cutin_road')
odr.add_road(road)
odr.adjust_roads_and_lanes()
odr.write_xml('cutin.xodr')
print('wrote cutin.xodr')

# 车辆
bb = OSC.BoundingBox(2.0, 4.8, 1.8, 1.4, 0.0, 0.9)
fa = OSC.Axle(2.5, 2.5, 1.8, 1.4, 0.9)
ba = OSC.Axle(0.0, 0.0, 1.8, 1.4, 0.9)
ego_veh = OSC.Vehicle('EgoCar', OSC.VehicleCategory.car, bb, fa, ba, 150, 10, 8)
tgt_veh = OSC.Vehicle('TargetCar', OSC.VehicleCategory.car, bb, fa, ba, 150, 10, 8)
entities = OSC.Entities()
entities.add_scenario_object('Ego', ego_veh)
entities.add_scenario_object('Target', tgt_veh)

# 初始状态
TD = OSC.TransitionDynamics(OSC.DynamicsShapes.step, OSC.DynamicsDimension.rate, 1)
init = OSC.Init()
init.add_init_action('Ego', OSC.TeleportAction(OSC.LanePosition(10, 0, -1, 0)))
init.add_init_action('Ego', OSC.AbsoluteSpeedAction(15, TD))
init.add_init_action('Target', OSC.TeleportAction(OSC.LanePosition(45, 0, -2, 0)))
init.add_init_action('Target', OSC.AbsoluteSpeedAction(12, TD))

# t=5s 切入
trig = OSC.ValueTrigger('t5', 0, OSC.ConditionEdge.rising,
                        OSC.SimulationTimeCondition(5, OSC.Rule.greaterThan))
event = OSC.Event('cutin', OSC.Priority.overwrite)
event.add_trigger(trig)
lc = OSC.AbsoluteLaneChangeAction(-1, OSC.TransitionDynamics(
    OSC.DynamicsShapes.sinusoidal, OSC.DynamicsDimension.time, 3))
event.add_action('lanechange', lc)
man = OSC.Maneuver('cutin_man'); man.add_event(event)
mgroup = OSC.ManeuverGroup('mg'); mgroup.add_actor('Target'); mgroup.add_maneuver(man)
act = OSC.Act(1, 'cutin_act'); act.add_maneuver_group(mgroup)
story = OSC.Story('cutin_story'); story.add_act(act)
sbd = OSC.StoryBoard(init, OSC.StopTrigger('stop'))
sbd.add_story(story)
sce = OSC.Scenario('cutin', 'hanks', OSC.ParameterDeclarations(),
                   entities, sbd, OSC.RoadNetwork('cutin.xodr'),
                   OSC.CatalogLocations(), OSC.Environment())
sce.write_xml('cutin.xosc')
print('wrote cutin.xosc')

## 2. esmini 无头播放
记录每 0.5s 的车辆位置，验证 Target 确实在 t≈5s 开始变道。

In [ ]:
import subprocess, re
r = subprocess.run(
    ['/opt/esmini/esmini/bin/esmini', '--headless',
     '--osc', 'cutin.xosc', '--road', 'cutin.xodr',
     '--fixed_timestep', '0.05'],
    capture_output=True, text=True, timeout=60)
print('returncode:', r.returncode)
# 找碰撞/结束信息
for line in (r.stdout + r.stderr).splitlines():
    if 'collision' in line.lower() or 'finish' in line.lower():
        print(line.strip())
print('done')

## 思考题
1. 把 Target 初速改为 18m/s（比 Ego 快），cut-in 还危险吗？为什么？
2. 如果你是 Ego 的 AEB 算法，看到 Target 开始变道时，最晚什么时候必须刹车？
3. 试着把变道时间从 3s 改为 1.5s（更急的加塞），观察结果变化。